Se implementa la función hash que calcula un valor hash intermedio sumando los valores ASCII de cada caracter de la clave y luego aplicandole módulo con el tamaño de la lista.

In [12]:
import math

# --- CAPA 1: GENERADORES DE HASH CODE (De Objeto a Entero) ---

def hash_code_ascii_naive(key):
    """Suma básica de caracteres ASCII."""
    return sum(ord(char) for char in key)

def hash_code_polynomial_31(key):
    """Hash polinomial estándar (Multiplicación por primos)."""
    h = 0
    acc = len(key)
    for char in key:
        h += ord(char) * pow(31, acc)
        acc -= 1
    return h


# --- CAPA 2: MÉTODOS DE COMPRESIÓN (De Entero a Índice de Tabla) ---

def compress_division(hash_func, key, size):
    """Método de la división clásico (Módulo)."""
    return hash_func(key) % size

def compress_multiplication_fallo(hash_func, key, size):
    A = 0.618033988749895
    h = hash_func(key) # Devuelve un entero inmenso, ej: 10^21 
    #falla porque no se puede representar con precisión decimal en Python
    return math.floor(size * ((h * A) % 1)) #este modulo 1 da siempre 0, porque h*A es un numero tan grande que pierde la parte decimal

def compress_multiplication(hash_func, key, size):
    """Método de la multiplicación de Knuth (Corregido para desbordamiento)."""
    A = 0.618033988749895  # Inverso de la proporción áurea
    # Acotamos a 32 bits para preservar la precisión decimal en Python
    hash_32bit = hash_func(key) % (2**32)
    return math.floor(size * ((hash_32bit * A) % 1))

def compress_multiplication_bitwise(hash_func, key, size):
    # 'p' es la potencia de 2. Si size es 128, p es 7 (porque 2^7 = 128)
    p = int(math.log2(size)) 
    
    h = hash_func(key)
    # Acotamiento a 32 bits usando máscara de bits (más rápido que el módulo)
    hash_32bit = h & 0xFFFFFFFF 
    
    # Constante entera de Knuth (A * 2^32)
    s = 2654435769 
    
    # Producto y extracción directa a 32 bits
    producto = (hash_32bit * s) & 0xFFFFFFFF
    
    # Desplazamiento a la derecha para obtener el índice
    return producto >> (32 - p)

Generando datos de prueba y evaluación de tablas mediante prueba chi cuadrado. Ojo la tabla hash no se genera ni se almacenan en un vector, ya que hay muchas colisiones y solo se está estudiando la distribución de los datos:

In [13]:
palabras_prueba = [f"usuario_i_{i}" for i in range(1000)]
N = len(palabras_prueba)


def evaluar_tabla(funcion_hash, f_hash, size):
    casillas = dict()
    for palabra in palabras_prueba:
        index = f_hash(funcion_hash, palabra, size)
        if index in casillas:
            casillas[index] += 1
        else:
            casillas[index] = 1
    esperado_por_casilla = N / size
    chi_cuadrado = 0
    for i in range(size):
        if i not in casillas:
            casillas[i] = 0
    chi_cuadrado = sum([(casillas[i] - esperado_por_casilla)**2 / esperado_por_casilla for i in range(size)])
    return chi_cuadrado, casillas



Medir velocidad:

In [14]:
import time
import math


def medir_velocidad(funcion_hash, f_hash, size):
    inicio = time.perf_counter()
    for palabra in palabras_prueba:
        f_hash(funcion_hash, palabra, size)
    fin = time.perf_counter()
    tiempo_ms = (fin - inicio) * 1000
    return tiempo_ms

Pruebas y mediciones de diversas implementaciones en cuanto a distribución de datos:

In [15]:
# =====================================================================
# BLOQUE DE EJECUCIÓN: PRUEBAS Y VARIANTES ESTRATÉGICAS
# =====================================================================

# 1. Ejecución de experimentos cruzando Capa 1 y Capa 2

# Experimentos Base (Con los nombres nuevos y limpios)
chi_naive_div_100, casillas_naive_div_100 = evaluar_tabla(hash_code_ascii_naive, compress_division, 100)
chi_naive_div_97,  casillas_naive_div_97  = evaluar_tabla(hash_code_ascii_naive, compress_division, 97)
chi_poly_div_97,   casillas_poly_div_97   = evaluar_tabla(hash_code_polynomial_31, compress_division, 97)

# VARIANTE 1: ¿Un buen Hash Code (Polinomial) salva a un tamaño compuesto (100) en División?
chi_poly_div_100, casillas_poly_div_100 = evaluar_tabla(hash_code_polynomial_31, compress_division, 100)

# VARIANTE 2: La prueba reina de la Multiplicación usando un tamaño Potencia de 2 (128)
chi_poly_mult_128, casillas_poly_mult_128 = evaluar_tabla(hash_code_polynomial_31, compress_multiplication, 128)

chi_poly_mult_bitwise_128, casillas_poly_mult_bitwise_128 = evaluar_tabla(hash_code_polynomial_31, compress_multiplication_bitwise, 128)
# 2. Impresión de los resultados del Laboratorio

chi_poly_mult_error_128, casillas_poly_mult_error_128 = evaluar_tabla(hash_code_polynomial_31, compress_multiplication_fallo, 128)

print("\n=================== RESULTADOS DEL LABORATORIO ===================")
print(f"1. Naive ASCII + División       (Tamaño Compuesto 100): Chi-cuadrado = {chi_naive_div_100:.2f}")
print(f"2. Naive ASCII + División       (Tamaño Primo 97):      Chi-cuadrado = {chi_naive_div_97:.2f}")
print(f"3. Polinomial  + División       (Tamaño Primo 97):      Chi-cuadrado = {chi_poly_div_97:.2f}")
print(f"4. Polinomial  + División       (Tamaño Compuesto 100): Chi-cuadrado = {chi_poly_div_100:.2f}")
print(f"5. Polinomial  + Multiplicación (Potencia de 2 - 128):  Chi-cuadrado = {chi_poly_mult_128:.2f}")
print(f"6. Polinomial  + Multiplicación Bitwise (Potencia de 2 - 128):  Chi-cuadrado = {chi_poly_mult_bitwise_128:.2f}")
print(f"7. Polinomial  + Multiplicación con Error (Potencia de 2 - 128):  Chi-cuadrado = {chi_poly_mult_error_128:.2f}")
print("==================================================================\n")

# Inspección visual de las distribuciones más óptimas
print("Distribución Variante 4 (Polinomial + División 100):")
print(casillas_poly_div_100)

print("\nDistribución Variante 5 (Polinomial + Multiplicación 128):")
print(casillas_poly_mult_128)

print("\nDistribución Variante 6 (Polinomial + Multiplicación Bitwise 128):")
print(casillas_poly_mult_bitwise_128)


=================== RESULTADOS DEL LABORATORIO ===================
1. Naive ASCII + División       (Tamaño Compuesto 100): Chi-cuadrado = 3763.20
2. Naive ASCII + División       (Tamaño Primo 97):      Chi-cuadrado = 3584.22
3. Polinomial  + División       (Tamaño Primo 97):      Chi-cuadrado = 29.56
4. Polinomial  + División       (Tamaño Compuesto 100): Chi-cuadrado = 24.00
5. Polinomial  + Multiplicación (Potencia de 2 - 128):  Chi-cuadrado = 31.68
6. Polinomial  + Multiplicación Bitwise (Potencia de 2 - 128):  Chi-cuadrado = 28.35
7. Polinomial  + Multiplicación con Error (Potencia de 2 - 128):  Chi-cuadrado = 127000.00

Distribución Variante 4 (Polinomial + División 100):
{79: 11, 10: 10, 41: 9, 72: 10, 3: 13, 34: 13, 65: 13, 96: 12, 27: 10, 58: 8, 98: 10, 29: 11, 60: 13, 91: 11, 22: 10, 53: 10, 84: 9, 15: 9, 46: 10, 77: 11, 59: 9, 90: 11, 21: 13, 52: 11, 83: 11, 14: 11, 45: 10, 76: 10, 7: 11, 38: 12, 20: 9, 51: 11, 82: 11, 13: 9, 44: 9, 75: 9, 6: 8, 37: 9, 68: 10, 99: 12, 81: 10

Pruebas y mediciones en cuanto a velocidad

In [16]:
# =====================================================================
# 2. BLOQUE DE EJECUCIÓN: PRUEBAS DE VELOCIDAD
# =====================================================================

# Experimentos Base
tiempo_naive_div_100 = medir_velocidad(hash_code_ascii_naive, compress_division, 100)
tiempo_naive_div_97  = medir_velocidad(hash_code_ascii_naive, compress_division, 97)
tiempo_poly_div_97   = medir_velocidad(hash_code_polynomial_31, compress_division, 97)

# VARIANTE 1
tiempo_poly_div_100 = medir_velocidad(hash_code_polynomial_31, compress_division, 100)

# VARIANTE 2 (Multiplicación con Floats y math.floor)
tiempo_poly_mult_128 = medir_velocidad(hash_code_polynomial_31, compress_multiplication, 128)

tiemo_poly_mult_bitwise_128 = medir_velocidad(hash_code_polynomial_31, compress_multiplication_bitwise, 128)

tiempo_poly_mult_error_128 = medir_velocidad(hash_code_polynomial_31, compress_multiplication_fallo, 128)



# =====================================================================
# 3. REPORTE DEL LABORATORIO DE RENDIMIENTO
# =====================================================================

print("\n=================== RESULTADOS DE VELOCIDAD (1000 LLAVES) ===================")
print(f"1. Naive ASCII + División       (Tamaño Compuesto 100): {tiempo_naive_div_100:.4f} ms")
print(f"2. Naive ASCII + División       (Tamaño Primo 97):      {tiempo_naive_div_97:.4f} ms")
print(f"3. Polinomial  + División       (Tamaño Primo 97):      {tiempo_poly_div_97:.4f} ms")
print(f"4. Polinomial  + División       (Tamaño Compuesto 100): {tiempo_poly_div_100:.4f} ms")
print(f"5. Polinomial  + Multiplicación (Potencia de 2 - Float):{tiempo_poly_mult_128:.4f} ms")
print(f"6. Polinomial  + Multiplicación Bitwise (Potencia de 2 - Bitwise): {tiemo_poly_mult_bitwise_128:.4f} ms")
print(f"7. Polinomial  + Multiplicación con Error (Potencia de 2 - Float con Error): {tiempo_poly_mult_error_128:.4f} ms")
print("=============================================================================\n")


=================== RESULTADOS DE VELOCIDAD (1000 LLAVES) ===================
1. Naive ASCII + División       (Tamaño Compuesto 100): 1.6799 ms
2. Naive ASCII + División       (Tamaño Primo 97):      1.8629 ms
3. Polinomial  + División       (Tamaño Primo 97):      3.7292 ms
4. Polinomial  + División       (Tamaño Compuesto 100): 5.2459 ms
5. Polinomial  + Multiplicación (Potencia de 2 - Float):4.3813 ms
6. Polinomial  + Multiplicación Bitwise (Potencia de 2 - Bitwise): 5.5829 ms
7. Polinomial  + Multiplicación con Error (Potencia de 2 - Float con Error): 4.3820 ms

